# Phase 2.3: LSTM Training - 1h Horizon (4 timesteps)

## Objectif
Entraîner LSTM pour prédiction court terme (1h, 4 timesteps).

## Architecture
- Input: (batch, 60, n_pollutants)
- LSTM 64 + Dropout 0.2 + return_sequences=True
- LSTM 32 + Dropout 0.2
- Dense 16 + ReLU
- Dense 4 (output)

## Sortie
- `ia/models/model_lstm_1h.h5`: Modèle Keras
- `ia/models/lstm_1h_metrics.json`: Métriques (RMSE, MAE, MAPE, R²)

## Section 1: Setup et Chargement Tenseurs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import pickle

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
tf.random.set_seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✅ Imports réussis")
print(f"   TensorFlow version: {tf.__version__}")

In [ ]:
# Configuration des chemins
PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

tensors_file = DATA_DIR / "lstm_train_val_test.pkl"
model_file = MODELS_DIR / "model_lstm_1h.h5"
metrics_file = MODELS_DIR / "lstm_1h_metrics.json"
history_file = MODELS_DIR / "lstm_1h_history.json"

print(f"📦 Tensors input: {tensors_file}")
print(f"💾 Model output: {model_file}")
print(f"📊 Metrics output: {metrics_file}")

In [ ]:
# Charger tenseurs préparés
print("🔄 Chargement tenseurs...\n")

with open(tensors_file, 'rb') as f:
    lstm_data = pickle.load(f)

# Extraire tenseurs pour horizon 1h (4 timesteps)
HORIZON = 4
combined = lstm_data['combined_tensors'][HORIZON]

X_train = combined['train'][0]
y_train = combined['train'][1]
X_val = combined['val'][0]
y_val = combined['val'][1]
X_test = combined['test'][0]
y_test = combined['test'][1]

print(f"✅ Tenseurs chargés pour horizon {HORIZON} (1h)")
print(f"\n   Dimensions:")
print(f"   X_train: {X_train.shape} (batch, lookback={X_train.shape[1]}, pollutants={X_train.shape[2]})")
print(f"   y_train: {y_train.shape} (batch, horizon={y_train.shape[1]})")
print(f"   X_val: {X_val.shape}")
print(f"   y_val: {y_val.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")

## Section 2: Architecture LSTM

In [ ]:
# ========================================
# SECTION 2: Construction Architecture LSTM
# ========================================
# LSTM = Long Short-Term Memory: réseau de neurones pour les séquences temporelles
# Avantage: peut apprendre les dépendances à long terme

print("🏗️  Construction architecture LSTM...\\n")

# Architecture:
# Input → LSTM(64) → Dropout → LSTM(32) → Dropout → Dense(16) → Dense(4) → Output
#
# Explications:
# - LSTM(64): 64 cellules LSTM pour capturer les patterns complexes
# - return_sequences=True: garder toutes les sorties temporelles (pas juste la dernière)
# - Dropout(0.2): désactiver 20% des neurones pour éviter overfitting
# - Dense(16): couche dense intermédiaire (réduction dimensionnelle)
# - Dense(4): sortie = 4 timesteps (≈ 1 heure)

model = keras.Sequential([
    # Couche d'entrée: (batch, lookback=60, n_pollutants=8)
    layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
    
    # Première couche LSTM: 64 unités
    # - return_sequences=True: passer toutes les sorties à la couche suivante
    # - Dropout réduit l'overfitting en "jetant" aléatoirement des neurones
    layers.LSTM(64, return_sequences=True, activation='relu'),
    layers.Dropout(0.2),
    
    # Deuxième couche LSTM: 32 unités (réduction)
    # - return_sequences=False (défaut): juste la dernière sortie
    layers.LSTM(32, activation='relu'),
    layers.Dropout(0.2),
    
    # Couches denses: apprentissage des patterns
    layers.Dense(16, activation='relu'),   # 16 neurones intermédiaires
    layers.Dense(HORIZON)                   # HORIZON=4 pour 1h (sortie)
])

# Compiler le modèle
# - loss='mse': Mean Squared Error (erreur moyenne au carré)
# - Adam: optimiseur adaptatif (meilleur que SGD classique)
# - learning_rate=0.001: pas d'apprentissage (petit = stable, grand = rapide)
model.compile(
    loss='mse',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['mae']  # Mean Absolute Error (métrique supp.)
)

print(f"✅ Modèle construit")
print(f"   Architecture: LSTM(64) → LSTM(32) → Dense(16) → Dense({HORIZON})")
print(f"   Paramètres: {model.count_params():,}")
print(f"\nSummaire du modèle:")
model.summary()

## Section 3: Entraînement

In [ ]:
# ========================================
# SECTION 3: Entraînement du modèle
# ========================================
# Faire apprendre au modèle LSTM les patterns dans les données

print("🔄 Entraînement LSTM (horizon 1h)...\\n")

# Early stopping: arrêter l'entraînement si la validation ne s'améliore plus
# Évite l'overfitting (apprentissage sur bruit de training au lieu de patterns généraux)
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',      # Observer la perte de validation
    patience=10,             # Attendre 10 epochs sans amélioration avant arrêt
    restore_best_weights=True,  # Restaurer les poids du meilleur epoch
    verbose=1
)

# Entraîner le modèle
# - X_train, y_train: données d'entraînement
# - validation_data: données pour valider et early stopping
# - epochs: max 100 itérations sur le dataset
# - batch_size: traiter 32 samples à la fois (compromis mémoire/vitesse)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print(f"\\n✅ Entraînement terminé après {len(history.history['loss'])} epochs")

In [ ]:
# Visualiser courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('LSTM 1h - Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_title('LSTM 1h - MAE', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'lstm_1h_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Courbes d'entraînement visualisées")

## Section 4: Évaluation

In [ ]:
# Prédictions et métriques
print("🔍 Évaluation sur test set...\n")

y_test_pred = model.predict(X_test)

# Calcul des métriques
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_test_pred)
mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100
r2 = r2_score(y_test, y_test_pred)

print(f"✅ Métriques sur test set:")
print(f"   RMSE: {rmse:.4f}")
print(f"   MAE:  {mae:.4f}")
print(f"   MAPE: {mape:.2f}%")
print(f"   R²:   {r2:.4f}")

# Targets de la spec
print(f"\n   Validation vs targets:")
print(f"   RMSE < 10%: {'✅' if rmse < 0.1 else '❌'}")
print(f"   R² > 0.85: {'✅' if r2 > 0.85 else '⚠️'}")
print(f"   MAPE < 10%: {'✅' if mape < 10 else '⚠️'}")

In [ ]:
# Visualiser prédictions vs réel (premiers 200 samples)
n_plot = min(200, len(y_test))

plt.figure(figsize=(14, 6))
for i in range(min(4, HORIZON)):
    plt.plot(y_test[:n_plot, i], label=f'Real (t+{i+1})', linewidth=1, alpha=0.7)
    plt.plot(y_test_pred[:n_plot, i], label=f'Pred (t+{i+1})', linewidth=1, alpha=0.7, linestyle='--')

plt.title(f'LSTM 1h - Prédictions vs Réel (premiers {n_plot} samples)', fontsize=12, fontweight='bold')
plt.xlabel('Sample')
plt.ylabel('Valeur normalisée')
plt.legend(loc='best', ncol=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'lstm_1h_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Visualisation prédictions sauvegardée")

## Section 5: Sauvegarde

In [ ]:
# Sauvegarder modèle
print("💾 Sauvegarde du modèle...\n")

model.save(model_file)
print(f"✅ Modèle sauvegardé: {model_file}")

# Sauvegarder metrics
metrics = {
    'horizon': 4,
    'horizon_hours': 1,
    'rmse': float(rmse),
    'mae': float(mae),
    'mape': float(mape),
    'r2_score': float(r2),
    'test_samples': len(y_test),
    'epochs_trained': len(history.history['loss'])
}

with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✅ Métriques sauvegardées: {metrics_file}")

# Sauvegarder history
history_dict = {
    'loss': history.history['loss'],
    'val_loss': history.history['val_loss'],
    'mae': history.history['mae'],
    'val_mae': history.history['val_mae']
}

with open(history_file, 'w') as f:
    json.dump(history_dict, f, indent=2)

print(f"✅ History sauvegardé: {history_file}")

## ✅ Résumé - LSTM 1h (Horizon 4)

✅ Modèle LSTM construit: LSTM(64) → LSTM(32) → Dense(16) → Dense(4)
✅ Entraîné sur 70% des données (lookback=60)
✅ Validé sur 15%, testé sur 15% (split temporel strict)
✅ Early stopping appliqué (patience=10)
✅ Métriques finales:
   - RMSE: {rmse:.4f}
   - MAE: {mae:.4f}
   - MAPE: {mape:.2f}%
   - R²: {r2:.4f}
✅ Modèle sauvegardé (.h5 format Keras)

**Prochaine étape**: Notebook 07 - LSTM 24h (horizon 96)